## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-19-16-53-29 +0000,bbc,Susan Smith denied parole again 32 years after...,https://www.bbc.co.uk/news/articles/ckdd1y09eq...
1,2026-11-19-16-53-00 +0000,wsj,Anthropic’s IPO Is a Win for Venture. Its Succ...,https://www.wsj.com/pro/venture-capital/anthro...
2,2026-11-19-16-51-56 +0000,nyt,"After Bridge Attack, Putin Says He Will ‘Not B...",https://www.nytimes.com/2026/11/19/world/europ...
3,2026-11-19-16-46-54 +0000,nyt,"Trump, Xi and Putin Seek to Project Power at T...",https://www.nytimes.com/live/2026/11/18/world/...
4,2026-11-19-16-46-27 +0000,nyt,Ericka Lomick’s Texas House Win Faces Challeng...,https://www.nytimes.com/2026/11/18/us/texas-ho...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2650/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
24,trump,58
8,anthropic,34
17,putin,26
276,new,23
25,xi,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
434,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...,156
244,2026-11-19-03-53-50 +0000,chitri,President Trump meets Xi and Putin in China wi...,https://www.chicagotribune.com/2026/11/18/trum...,154
257,2026-11-19-02-44-01 +0000,nypost,Kim Jong Un’s nuclear program draws ‘serious c...,https://nypost.com/2026/11/18/world-news/kim-j...,147
191,2026-11-19-10-24-31 +0000,latimes,"'Making history': Trump, Putin and Xi meet in ...",https://www.latimes.com/politics/story/2026-11...,145
365,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...,145


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
434,156,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...
262,90,2026-11-19-02-00-00 +0000,wsj,Jensen Huang’s ouster from Nvidia is giving ne...,https://www.wsj.com/politics/national-security...
19,89,2026-11-19-16-19-23 +0000,bbc,Myanmar committed genocide against Rohingya mi...,https://www.bbc.co.uk/news/articles/c804wzp20d...
348,82,2026-11-18-22-13-12 +0000,cbc,She says Russell Brand sexually assaulted her ...,https://www.cbc.ca/radio/asithappens/mia-latte...
7,81,2026-11-19-16-41-29 +0000,nypost,Anthropic begins trading at $111.50 per share ...,https://nypost.com/2026/11/19/business/anthrop...
277,74,2026-11-19-01-11-00 +0000,wsj,Elon Musk said he felt “pretty foolish” about ...,https://www.wsj.com/politics/policy/elon-musk-...
384,68,2026-11-18-20-56-05 +0000,wapo,Democrat Hakeem Jeffries nominated to be first...,https://www.washingtonpost.com/politics/2026/1...
176,50,2026-11-19-11-00-00 +0000,latimes,"High tides, big storms: Next week will be key ...",https://www.latimes.com/california/story/2026-...
322,47,2026-11-18-23-09-09 +0000,nypost,Shocking video shows Putin’s prized Crimea bri...,https://nypost.com/2026/11/18/world-news/commu...
76,46,2026-11-19-14-55-00 +0000,wsj,Marketers Level Up: The Unignorable Power of t...,https://www.wsj.com/cmo-today/marketers-level-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
